# Improved CMJ–Pitching Correlation Analysis

**Public portfolio version. No athlete data included.** This is a methodological update, **not** a claim that these procedures were used in the original conference analysis. 

**Design:** retain each individual pitch; average repeated CMJ trials within athlete and calendar date; join only same-day CMJ to that athlete's pitches; calculate exploratory pitch-level Pearson correlations separately by athlete and pitch type. No forward-filling across days. Repeated pitches share a CMJ assessment, so pitch-level correlations are descriptive and **pitch count is not independent CMJ sample size**. No naive pitch-level p-values are reported. This notebook cannot establish causal relationships or inferential significance.

**Check the schema first:** This version assumes `PitcherID` and `Pitcher` represent the same athlete, and that CMJ trials are athlete/day measurements (not pitch-type-specific). If your study protocol defines CMJ records by pitch type, or if the identifier systems differ, revise the join only after confirming the intended mapping. Column names in the lists below must match your spreadsheets; this notebook does not include real data.

In [ ]:
%pip -q install reportlab openpyxl

In [ ]:
from pathlib import Path
from html import escape
import numpy as np
import pandas as pd
from google.colab import files
from reportlab.lib import colors
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import SimpleDocTemplate, LongTable, TableStyle, Paragraph, Spacer, PageBreak

# Replace only in your private working copy; keep confidential team identifiers off GitHub.
TEAM_NAME = "YOUR_TEAM_NAME"
OUTPUT_PDF = Path('/content/correlation_results_improved.pdf')
OUTPUT_CSV = Path('/content/correlation_results_improved.csv')

PITCH_VARS = [
    'RelSpeed', 'VertRelAngle', 'HorzRelAngle', 'SpinRate', 'SpinAxis',
    'RelHeight', 'RelSide', 'Extension', 'VertBreak',
    'InducedVertBreak', 'HorzBreak'
]
CMJ_VARS = [
    'Concentric Duration', 'Concentric Mean Force',
    'Concentric Mean Power / BM', 'Concentric Peak Force',
    'Eccentric:Concentric Mean Force Ratio',
    'Eccentric Deceleration Phase Duration', 'Eccentric Duration',
    'Eccentric Deceleration Mean Force', 'Eccentric Peak Power',
    'Eccentric Peak Power / BW', 'Eccentric Peak Velocity'
]
# Adjust metric names above to the actual authorized input schema; do not publish private data.

In [ ]:
print('Upload the PITCHING Excel spreadsheet (.xlsx):')
pitch_upload = files.upload()
if len(pitch_upload) != 1:
    raise ValueError('Upload exactly one pitching spreadsheet.')
print('Upload the CMJ Excel spreadsheet (.xlsx):')
cmj_upload = files.upload()
if len(cmj_upload) != 1:
    raise ValueError('Upload exactly one CMJ spreadsheet.')
pitch_df = pd.read_excel(next(iter(pitch_upload)))
cmj_df = pd.read_excel(next(iter(cmj_upload)))
pitch_df.columns = pitch_df.columns.astype(str).str.strip()
cmj_df.columns = cmj_df.columns.astype(str).str.strip()
print(f'Loaded {len(pitch_df):,} pitching rows and {len(cmj_df):,} CMJ rows.')

In [ ]:
pitch_required = ['PitcherID', 'PitcherTeam', 'Date', 'TaggedPitchType', *PITCH_VARS]
cmj_required = ['Pitcher', 'Date', *CMJ_VARS]
for label, df, required in [('Pitching', pitch_df, pitch_required), ('CMJ', cmj_df, cmj_required)]:
    if df.columns.duplicated().any():
        raise ValueError(f'{label}: duplicate column names after stripping whitespace.')
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'{label}: missing required columns: {missing}')
if TEAM_NAME == 'YOUR_TEAM_NAME':
    raise ValueError('Set TEAM_NAME in your private working copy before running.')

pitch_df = pitch_df.loc[pitch_df['PitcherTeam'].astype('string').str.strip().eq(TEAM_NAME)].copy()
if pitch_df.empty:
    raise ValueError('No pitching rows match TEAM_NAME. Check your private configuration.')

# Use calendar dates. invalid dates become missing and are counted below.
for df in (pitch_df, cmj_df):
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.normalize()

# Match identifiers as strings. strip whitespace and normalize Excel integer-like IDs.
def normalized_id(series):
    s = series.astype('string').str.strip()
    return s.str.replace(r'^(\d+)\.0$', r'\1', regex=True).replace('', pd.NA)

pitch_df['_athlete_key'] = normalized_id(pitch_df['PitcherID'])
cmj_df['_athlete_key'] = normalized_id(cmj_df['Pitcher'])
pitch_df['TaggedPitchType'] = pitch_df['TaggedPitchType'].astype('string').str.strip().replace('', pd.NA)

for metric in PITCH_VARS:
    pitch_df[metric] = pd.to_numeric(pitch_df[metric], errors='coerce')
for metric in CMJ_VARS:
    cmj_df[metric] = pd.to_numeric(cmj_df[metric], errors='coerce')

pitch_usable = pitch_df.dropna(subset=['_athlete_key', 'Date', 'TaggedPitchType']).copy()
cmj_usable = cmj_df.dropna(subset=['_athlete_key', 'Date']).copy()
print(f'Pitch rows excluded for missing ID/date/type: {len(pitch_df)-len(pitch_usable):,}')
print(f'CMJ rows excluded for missing ID/date: {len(cmj_df)-len(cmj_usable):,}')

# CMJ is assumed to be an athlete/day assessment, not a pitch-type-specific measurement.
cmj_daily = (cmj_usable.groupby(['_athlete_key', 'Date'], as_index=False)[CMJ_VARS]
             .mean())
if cmj_daily.duplicated(['_athlete_key', 'Date']).any():
    raise ValueError('CMJ daily keys are not unique; review grouping.')

merged = pitch_usable.merge(
    cmj_daily, on=['_athlete_key', 'Date'], how='left',
    validate='many_to_one', indicator=True, suffixes=('_pitch', '_cmj')
)
print(f'Eligible individual pitches: {len(pitch_usable):,}')
print(f'Pitches matched to a same-day CMJ assessment: {(merged["_merge"] == "both").sum():,}')
print(f'Pitches without a same-day CMJ assessment: {(merged["_merge"] != "both").sum():,}')
matched = merged.loc[merged['_merge'].eq('both')].copy()
if matched.empty:
    raise ValueError('No same-day athlete matches. Verify dates, identifier mapping, and CMJ schema.')
if len(merged) != len(pitch_usable):
    raise AssertionError('Unexpected row multiplication during merge.')
print(f'Unique matched athlete-days: {matched[["_athlete_key", "Date"]].drop_duplicates().shape[0]:,}')

In [ ]:
# Each coefficient uses complete pairs for that metric combination.
# n_pitches counts pitch rows; n_cmj_days counts distinct dates contributing CMJ data.
results = []
for (athlete, pitch_type), group in matched.groupby(['_athlete_key', 'TaggedPitchType'], sort=True):
    for p_var in PITCH_VARS:
        for c_var in CMJ_VARS:
            pair = group[['Date', p_var, c_var]].dropna()
            n_pitches = len(pair)
            n_cmj_days = pair['Date'].nunique()
            if n_pitches < 3 or n_cmj_days < 3:
                r, status = np.nan, 'Insufficient data (<3 pitches or <3 CMJ days)'
            elif pair[p_var].nunique() < 2 or pair[c_var].nunique() < 2:
                r, status = np.nan, 'Constant metric'
            else:
                r = pair[p_var].corr(pair[c_var], method='pearson')
                status = 'Descriptive only' if pd.notna(r) else 'Undefined correlation'
            results.append({
                'Athlete_ID': athlete, 'Pitch_Type': pitch_type,
                'Pitch_Metric': p_var, 'CMJ_Metric': c_var,
                'Pearson_r': r, 'N_pitches': n_pitches,
                'N_CMJ_days': n_cmj_days, 'Status': status
            })
results_df = pd.DataFrame(results)
if results_df.empty:
    raise ValueError('No pitcher/pitch-type groups available after matching.')
results_df.to_csv(OUTPUT_CSV, index=False)
print(f'Calculated {len(results_df):,} metric pairs across {results_df[["Athlete_ID", "Pitch_Type"]].drop_duplicates().shape[0]:,} athlete/pitch-type groups.')
print('CSV saved locally; do not commit real-data outputs to GitHub.')

In [ ]:
# PDF uses landscape pages and wrapping text for long metric names.
styles = getSampleStyleSheet()
small = ParagraphStyle('SmallCell', parent=styles['Normal'], fontName='Helvetica',
                       fontSize=7, leading=9, alignment=TA_LEFT)
heading = ParagraphStyle('ReportHeading', parent=styles['Heading2'], spaceBefore=9, spaceAfter=7)
doc = SimpleDocTemplate(str(OUTPUT_PDF), pagesize=landscape(A4),
                        leftMargin=12*mm, rightMargin=12*mm,
                        topMargin=12*mm, bottomMargin=12*mm)
story = [Paragraph('CMJ–Pitching Correlation Analysis (Improved)', styles['Title']),
         Paragraph('Exploratory pitch-level Pearson correlations. Individual pitches are retained, '
                   'but multiple pitches may share the same daily CMJ assessment. '
                   'N_pitches is not the number of independent CMJ observations; '
                   'no inferential p-values are reported.', styles['Normal']), Spacer(1, 8)]
headers = ['Pitch metric', 'CMJ metric', 'Pearson r', 'N pitches', 'N CMJ days', 'Status']
widths = [38*mm, 65*mm, 22*mm, 20*mm, 23*mm, 82*mm]
for (athlete, pitch_type), section in results_df.groupby(['Athlete_ID', 'Pitch_Type'], sort=True):
    story.append(Paragraph(f'Athlete: {escape(str(athlete))} | Pitch type: {escape(str(pitch_type))}', heading))
    rows = [[Paragraph(escape(h), small) for h in headers]]
    for row in section.itertuples(index=False):
        r_text = 'N/A' if pd.isna(row.Pearson_r) else f'{row.Pearson_r:.3f}'
        values = [row.Pitch_Metric, row.CMJ_Metric, r_text, str(row.N_pitches),
                  str(row.N_CMJ_days), row.Status]
        rows.append([Paragraph(escape(str(v)), small) for v in values])
    table = LongTable(rows, colWidths=widths, repeatRows=1, hAlign='LEFT')
    table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.lightgrey),
        ('GRID', (0, 0), (-1, -1), 0.2, colors.grey),
        ('VALIGN', (0, 0), (-1, -1), 'TOP'),
        ('TOPPADDING', (0, 0), (-1, -1), 3),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 3),
    ]))
    story.extend([table, Spacer(1, 9)])
doc.build(story)
print('PDF saved locally; do not commit real-data outputs to GitHub.')
files.download(str(OUTPUT_CSV))
files.download(str(OUTPUT_PDF))

## Interpretation and publication notes

- This notebook does **not** forward-fill across dates or average individual pitches. It averages repeated CMJ trials within an athlete/day and uses the resulting same-day value for each pitch.
- `N_CMJ_days` is the count of distinct matched calendar dates **for each metric pair**; it is not a formal effective sample size. Multiple athletes are analyzed separately; repeated pitch observations and potentially serially correlated CMJ assessments require a suitable repeated-measures model for formal inference.
- The `<3` threshold is only a minimal computational guard, **not** a scientific adequacy criterion. Interpret correlations with very few testing days cautiously.
- The original uploaded notebook's CMJ metric labels omit some units; verify the exact headers in the authorized spreadsheets before running. Do not commit real spreadsheets, outputs, athlete IDs, or reports to a public repository.